# Comments

Comments are not there to narrate obvious syntax. In engineering systems, comments are part of maintainability strategy: they capture intent, invariants, non-obvious tradeoffs, and operational constraints that code alone cannot reliably communicate.


## 1. Problem First

Why does this matter in real systems?

- A retry loop exists because of a vendor rate-limit bug, but no one documents it.
- A parsing rule looks wrong, yet it preserves backward compatibility for legacy logs.
- A stale comment sends the next engineer in the wrong debugging direction.


In [26]:
def should_retry(status_code):
    # Vendor occasionally returns 499 during deploy windows.
    # We treat it as transient to avoid dropping ingestion jobs.
    return status_code in {429, 499, 500, 502, 503, 504}

## 2. Minimal Concept

Core syntax:

- `#` starts a single-line comment.
- Triple-quoted strings are string literals, not comments.
- Docstrings document modules, classes, and functions for tooling and introspection.


## 3. Mental Model

How Python actually behaves

Comments are stripped by the tokenizer and do not become runtime objects. Docstrings are different: when placed as the first statement in a module, class, or function, they are stored and become accessible via `__doc__`.


In [29]:
def load_config(path):
    """Load config from disk and return a dictionary."""
    # In production this would validate schema before returning.
    return {"path": path}

print(load_config.__doc__)

Load config from disk and return a dictionary.


## 4. Internal Mechanics

Behavior proof:

- Comments disappear before bytecode generation.
- Docstrings remain as constants and are assigned to `__doc__`.


In [31]:
import dis

def sample():
    """Docstring survives."""
    # Comment does not survive.
    return 1

print(sample.__doc__)
dis.dis(sample)

Docstring survives.
  6           0 LOAD_CONST               1 (1)
              2 RETURN_VALUE


## 5. Edge Cases

Where it breaks:

- Outdated comments become lies.
- Commented-out code rots and confuses ownership.
- Triple-quoted strings used as fake comments still allocate as constants in some contexts.
- Explaining obvious lines adds noise instead of clarity.


In [33]:
def noisy_total(values):
    # Create a variable called total.
    total = 0
    # Loop over all values.
    for value in values:
        # Add value to total.
        total += value
    return total

print(noisy_total([1, 2, 3]))

6


## 6. Performance Thinking

Comments do not affect runtime performance directly, but they affect engineering throughput:

- Good comments reduce incident debugging time.
- Bad comments increase review and maintenance cost.
- Over-commenting raises cognitive load because signal gets buried in noise.


## 7. Real Use Case

In a log ingestion service, a tiny normalization rule may look arbitrary. A good comment explains the external system behavior that forced the choice.


In [35]:
def normalize_level(level):
    # Some old agents emit "WARN" while new ones emit "WARNING".
    # We normalize both so downstream aggregations remain stable.
    return "WARNING" if level == "WARN" else level

print(normalize_level("WARN"))

WARNING


## 8. Anti-Pattern

What beginners do wrong:

- Use comments to explain every obvious line.
- Leave dead code commented out instead of deleting it.
- Document behavior in comments but never enforce it in tests.


In [37]:
# Bad: the comment is doing work the code should make clearer.
def process(items):
    # if no items return empty list
    if not items:
        return []
    return [item.strip() for item in items]

## 9. Interview Signals

Questions FAANG asks:

- When should a comment exist instead of cleaner code?
- What is the difference between a comment and a docstring?
- Why is stale documentation dangerous in production systems?
- What kind of knowledge belongs in comments versus tests?


## 10. Exercise (Non-trivial)

You inherit a parser full of redundant comments and one crucial undocumented hack. Rewrite the comments so only non-obvious intent remains, and add one test for every comment that captures business-critical behavior.


In [39]:
def parse_line(line):
    # split line
    parts = line.split("|")
    # old mobile agents sometimes omit the level field, so we pad it
    if len(parts) == 2:
        parts.insert(1, "INFO")
    # return parts
    return parts

# Task:
# 1. Keep only comments that carry real system knowledge.
# 2. Improve the code shape.
# 3. Add tests for the mobile-agent edge case.